### Json Parsing And Processing

In [20]:
import json
import os

In [21]:
os.makedirs("data/json_files", exist_ok=True)

In [22]:
# Sample nested JSON data
json_data = {
    "company": "Khemshield",
    "employees": [
        {
            "id": 1,
            "name": "Abdulkareem Adamu",
            "role": "Security and AI Engineer",
            "skills": ["Python", "JavaScript", "SQL"],
            "projects": [
                {"name": "RAG System", "status": "In Progress"},
                {"name": "Data Pipeline", "status": "Completed"}
            ]
        },
        {
            "id": 2,
            "name": "Auwal Tijani",
            "role": "Data Scientist",
            "skills": ["Python", "Machine Learning", "SQL"],
            "projects": [
                {"name": "ML Model", "status": "In Progress"},
                {"name": "Analytics Dashboard", "status": "Planning"}
            ]
        }
    ],
    "departments": {
        "engineering": {
            "head": "Mike Johnson",
            "budget": 1000000,
            "team_size": 25
        },
        "data_science": {
            "head": "Sarah Williams",
            "budget": 750000,
            "team_size": 15
        }
    }
}

json_data

{'company': 'Khemshield',
 'employees': [{'id': 1,
   'name': 'Abdulkareem Adamu',
   'role': 'Security and AI Engineer',
   'skills': ['Python', 'JavaScript', 'SQL'],
   'projects': [{'name': 'RAG System', 'status': 'In Progress'},
    {'name': 'Data Pipeline', 'status': 'Completed'}]},
  {'id': 2,
   'name': 'Auwal Tijani',
   'role': 'Data Scientist',
   'skills': ['Python', 'Machine Learning', 'SQL'],
   'projects': [{'name': 'ML Model', 'status': 'In Progress'},
    {'name': 'Analytics Dashboard', 'status': 'Planning'}]}],
 'departments': {'engineering': {'head': 'Mike Johnson',
   'budget': 1000000,
   'team_size': 25},
  'data_science': {'head': 'Sarah Williams',
   'budget': 750000,
   'team_size': 15}}}

In [23]:
with open("data/json_files/company_data.json", "w") as f:
    json.dump(json_data, f, indent=2)

In [24]:
# Save JSON Lines format
jsonl_data = [
    {"timestamp": "2024-01-01", "event": "user_login", "user_id": 123},
    {"timestamp": "2024-01-01", "event": "page_view", "user_id": 123, "page": "/home"},
    {"timestamp": "2024-01-01", "event": "purchase", "user_id": 123, "amount": 99.99}
]

with open('data/json_files/events.jsonl', 'w') as f:
    for item in jsonl_data:
        f.write(json.dumps(item) + '\n')

## Json Processing Stratergies

In [25]:
from langchain_community.document_loaders import JSONLoader

#### Method 1 : JsonLoader With jq_schema

In [26]:
loader = JSONLoader(
    "data/json_files/company_data.json", 
    jq_schema=".",
    text_content=False,
)

In [27]:
loader.load()

[Document(metadata={'source': 'C:\\Users\\khem6\\Desktop\\Training\\rag-bootcamp-projects\\1-DataIngestParsing\\data\\json_files\\company_data.json', 'seq_num': 1}, page_content='{"company": "Khemshield", "employees": [{"id": 1, "name": "Abdulkareem Adamu", "role": "Security and AI Engineer", "skills": ["Python", "JavaScript", "SQL"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}, {"id": 2, "name": "Auwal Tijani", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "Analytics Dashboard", "status": "Planning"}]}], "departments": {"engineering": {"head": "Mike Johnson", "budget": 1000000, "team_size": 25}, "data_science": {"head": "Sarah Williams", "budget": 750000, "team_size": 15}}}')]

In [28]:
loader = JSONLoader(
    "data/json_files/company_data.json", 
    jq_schema=".employees[]",
    text_content=False,
)

loader.load()

[Document(metadata={'source': 'C:\\Users\\khem6\\Desktop\\Training\\rag-bootcamp-projects\\1-DataIngestParsing\\data\\json_files\\company_data.json', 'seq_num': 1}, page_content='{"id": 1, "name": "Abdulkareem Adamu", "role": "Security and AI Engineer", "skills": ["Python", "JavaScript", "SQL"], "projects": [{"name": "RAG System", "status": "In Progress"}, {"name": "Data Pipeline", "status": "Completed"}]}'),
 Document(metadata={'source': 'C:\\Users\\khem6\\Desktop\\Training\\rag-bootcamp-projects\\1-DataIngestParsing\\data\\json_files\\company_data.json', 'seq_num': 2}, page_content='{"id": 2, "name": "Auwal Tijani", "role": "Data Scientist", "skills": ["Python", "Machine Learning", "SQL"], "projects": [{"name": "ML Model", "status": "In Progress"}, {"name": "Analytics Dashboard", "status": "Planning"}]}')]

#### Method 2: Custom JSON processing for complex structures

In [29]:
from typing import List
from langchain_core.documents import Document

def process_json_intelligently(filepath: str) -> List[Document]:
    """Process JSON with intelligent flattening and context preservation"""
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    documents = []
    
    # Strategy 1: Create documents for each employee with full context
    for emp in data.get('employees', []):
        content = f"""Employee Profile:
        Name: {emp['name']}
        Role: {emp['role']}
        Skills: {', '.join(emp['skills'])}

        Projects:"""

        for proj in emp.get('projects', []):
            content += f"\n- {proj['name']} (Status: {proj['status']})"
        
        doc = Document(
            page_content=content,
            metadata={
                'source': filepath,
                'data_type': 'employee_profile',
                'employee_id': emp['id'],
                'employee_name': emp['name'],
                'role': emp['role']
            }
        )
        documents.append(doc)

    return documents

process_json_intelligently("data/json_files/company_data.json")

[Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 1, 'employee_name': 'Abdulkareem Adamu', 'role': 'Security and AI Engineer'}, page_content='Employee Profile:\n        Name: Abdulkareem Adamu\n        Role: Security and AI Engineer\n        Skills: Python, JavaScript, SQL\n\n        Projects:\n- RAG System (Status: In Progress)\n- Data Pipeline (Status: Completed)'),
 Document(metadata={'source': 'data/json_files/company_data.json', 'data_type': 'employee_profile', 'employee_id': 2, 'employee_name': 'Auwal Tijani', 'role': 'Data Scientist'}, page_content='Employee Profile:\n        Name: Auwal Tijani\n        Role: Data Scientist\n        Skills: Python, Machine Learning, SQL\n\n        Projects:\n- ML Model (Status: In Progress)\n- Analytics Dashboard (Status: Planning)')]